## 🅿️ Détection de Places de Parking

Ce notebook implémente un système de détection automatique des places de parking libres et occupées à partir d'une image aérienne, en utilisant le traitement d'image avec **OpenCV**.

**Pipeline :**
1. Définition des places (ROI)
2. Prétraitement de l'image
3. Détection des objets + Comptage des pixels


### Importation des bibliothèques

- `numpy` : calculs mathématiques sur les tableaux
- `cv2` (OpenCV) : traitement d'image
- `pickle` : sauvegarde et chargement des données


In [1]:
import numpy as np
import cv2
import pickle

---
## 1. Définition des places de parking (ROI)

On utilise la fonction **`cv2.selectROIs`** d'OpenCV pour définir manuellement les places de parking.

Cette méthode permet de **sélectionner des zones d'intérêt (ROI = Region Of Interest)** directement sur l'image à l'aide du curseur de la souris :
- On dessine un rectangle autour de chaque place
- On appuie sur **Entrée** pour valider et passer à la suivante
- On appuie sur **Échap** pour terminer la sélection

Les coordonnées `(x, y, largeur, hauteur)` de chaque rectangle sont ensuite sauvegardées dans un fichier `positions.pkl` pour être réutilisées sans avoir à resélectionner.


In [3]:
# Chargement de l'image du parking depuis le disque
img = cv2.imread('parking.jpg')
# Redimensionner l'image pour qu'elle s'affiche correctement à l'écran

img_resized = cv2.resize(img, (1280, 720))


In [ ]:

# Ouvrir une fenêtre interactive où on dessine les zones (ROI) avec la souris
# --> Appuyer sur ENTRÉE pour valider chaque ROI, ECHAP pour terminer
rois = cv2.selectROIs("parking", img_resized, showCrosshair=True, fromCenter=False)

# Convertir les ROI en liste de tuples (x, y, largeur, hauteur)
positions = [tuple(map(int, roi)) for roi in rois]

# Sauvegarder les positions dans un fichier pour les réutiliser plus tard
with open("positions.pkl", "wb") as f:
    pickle.dump(positions, f)

print(f"{len(positions)} places sauvegardées")
cv2.destroyAllWindows()

33 places sauvegardées


---
## 2. Prétraitement de l'image

Avant d'analyser chaque place, on prépare l'image pour faciliter la détection.
L'objectif est d'obtenir une **image binaire** (noir et blanc) où les objets ressortent clairement.


### 2.1 Conversion en niveaux de gris

On convertit l'image couleur (BGR) en **niveaux de gris**.
Cela réduit la complexité : au lieu de 3 canaux (Bleu, Vert, Rouge), on n'a plus qu'**un seul canal d'intensité** par pixel.


In [4]:
# Convertir l'image en niveaux de gris (plus simple à analyser que couleur)
gray_img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

### 2.2 Réduction du bruit (Flou Gaussien)

On applique un **flou gaussien** pour lisser l'image et supprimer les petits détails parasites (grains, textures du sol, ombres légères).
Cela améliore la qualité de la binarisation à l'étape suivante.


### 2.3 Binarisation adaptative

On transforme l'image en **noir et blanc** (binarisation) de façon adaptative :
- Le seuil est calculé **localement** pour chaque zone de l'image
- Cela permet de gérer les variations d'éclairage (ombres, reflets)
- Résultat : les objets (voitures) apparaissent en **blanc**, le fond en **noir**


In [5]:
# Appliquer un flou gaussien pour réduire le bruit et les détails inutiles
# (5,5) = taille du filtre, 1 = intensité du flou
img_filtree = cv2.GaussianBlur(gray_img, (5, 5), 1)

In [6]:
# Binarisation adaptative : transforme l'image en noir et blanc
# Chaque pixel devient blanc (255) s'il appartient à un objet, noir (0) sinon
# ADAPTIVE_THRESH_GAUSSIAN_C = seuil calculé localement selon le voisinage
# THRESH_BINARY_INV = inverser le résultat (objets = blanc, fond = noir)
img_binaire = cv2.adaptiveThreshold(
    img_filtree,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    25,   # taille du voisinage
    16    # constante soustraite à la moyenne
)

---
## 3. Détection des objets dans chaque place

Pour chaque place de parking définie à l'étape 1, on **extrait la zone correspondante** dans l'image binaire, puis on **compte le nombre de pixels blancs** (pixels non nuls).

- Si le nombre de pixels blancs est **inférieur au seuil** → la place est **libre** 🟢
- Si le nombre est **supérieur au seuil** → une voiture est présente, la place est **occupée** 🔴


In [7]:
# Recharger les positions sauvegardées depuis le fichier
with open("positions.pkl", "rb") as f:
    positions = pickle.load(f)

In [8]:
img_affichage1 = img_resized.copy()
seuil = 500  # nombre de pixels blancs au-dessus duquel on considère la place occupée

for (x, y, w, h) in positions:
    # Extraire la zone (ROI) correspondant à cette place
    roi = img_binaire[y:y+h, x:x+w]

    # Compter le nombre de pixels blancs dans la ROI
    # Plus il y a de pixels blancs, plus il y a d'objets détectés
    count = cv2.countNonZero(roi)

    # Comparer avec le seuil pour décider si la place est libre ou occupée
    if count < seuil:
        couleur = (0, 255, 0)   # vert = place libre
    else:
        couleur = (0, 0, 255)   # rouge = place occupée

    # Dessiner le rectangle coloré sur l'image d'affichage
    cv2.rectangle(img_affichage1, (x, y), (x+w, y+h), couleur, 2)

# Afficher le résultat final
cv2.imshow("resultat", img_affichage1)
cv2.waitKey(0)
cv2.destroyAllWindows()